**1. Preparación de Datos**

1.1 Selección del Conjunto de Datos

Para el desarrollo de esta practica se utilizó el conjunto de datos **Brazilian E-Commerce Public Dataset by Olist**, disponible en la plataforma Kaggle. Este dataset contiene información real sobre transacciones de comercio electrónico en Brasil y describe el ciclo completo de vida de los pedidos realizados en la plataforma Olist.

Con el objetivo de mantener el análisis enfocado y alineado con los requisitos de la práctica, se seleccionó exclusivamente el archivo **olist_orders_dataset.csv**, el cual concentra la información necesaria para representar procesos secuenciales y temporales. Este único conjunto de datos se emplea para la construcción tanto del Heatmap como del Funnel Chart, evitando la necesidad de realizar uniones entre múltiples tablas y simplificando el preprocesamiento.

La elección de este dataset se justifica porque permite modelar de forma clara y realista un proceso de negocio por etapas, condición fundamental para la correcta construcción de un gráfico de tipo funnel, asi como para el analisis de patrones temporales mediante mapas de calor.



1.2 Carga y Exploración Inicial de los Datos

El archivo **olist_orders_dataset.csv** fue cargado en el notebook de Python utilizando la librería pandas. Una vez cargados los datos, se realizó una exploración inicial para verificar su estructura, dimensiones y variables disponibles.

El conjunto de datos contiene una fila por pedido y las siguientes variables relevantes:

order_approved_at: fecha y hora de aprobación del pago.

order_delivered_carrier_date: fecha en que el pedido fue entregado al operador logístico.

order_delivered_customer_date: fecha de entrega efectiva al cliente.

order_estimated_delivery_date: fecha estimada de entrega informada al cliente.


*   **order_id**: identificador único del pedido.
*   **customer_id**: identificador del cliente asociado al pedido.
*   **order_status**: estado del pedido (por ejemplo: delivered, shipped, canceled).
*   **order_purchase_timestamp**: fecha y hora en que se realizó la compra.
*   **order_approved_at**: fecha y hora de aprobación del pago.
*   **order_delivered_carrier_date**: fecha en que el pedido fue entregado al operador logístico.
*   **order_delivered_customer_date**: fecha de entrega efectiva al cliente.
*   **order_estimated_delivery_date**: fecha estimada de entrega informada al cliente.

Desde el punto de vista del tipo de variables, se identifican las siguientes categorías:

Variables categóricas:

*   order_status

Variables temporales (tratadas como fechas):

*   order_purchase_timestamp
*   order_approved_at
*   order_delivered_carrier_date
*   order_delivered_customer_date
*   order_estimated_delivery_date

Variables identificadoras:

*   order_id
*   customer_id

Este conjunto de variables resulta adecuado para analizar tanto la evolución temporal de los pedidos como su progresión a través de distintas etapas del proceso logístico.

1.3 Preprocesamiento de los Datos

Con el fin de preparar los datos para su posterior visualización, se aplicaron una serie de pasos básicos de preprocesamiento.

En primer lugar, las columnas de fecha fueron convertidas al formato datetime, lo que permite realizar agregaciones temporales y comparaciones de manera correcta. Posteriormente, se filtraron aquellos registros que presentaban valores nulos en las variables clave asociadas a las etapas del proceso, ya que estos distorsionan la interpretación de los gráficos.

Para mejorar la claridad de las visualizaciones, se consideraron solo los estados de pedido relevantes para el análisis del flujo principal del proceso (por ejemplo, pedidos creados, enviados y entregados), excluyendo estados poco frecuentes o no representativos para el objetivo del ejercicio.

No se realizó un tratamiento de valores atípicos numéricos, dado que el análisis se centra principalmente en conteos y patrones temporales, y no en magnitudes continuas como precios o cantidades. Este nivel de preprocesamiento se considera suficiente para garantizar la calidad de los datos sin introducir transformaciones innecesarias que compliquen la interpretación visual.

Tras estos pasos, el conjunto de datos quedó correctamente preparado para la construcción de las visualizaciones dinámicas requeridas, asegurando coherencia temporal, claridad semántica y alineación con los objetivos de la práctica.

In [1]:
import pandas as pd
import plotly.express as px

In [2]:
df = pd.read_csv("/content/olist_orders_dataset.csv")

# Convertir columnas de tiempo a datetime
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for c in date_cols:
    df[c] = pd.to_datetime(df[c], errors="coerce")

df.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


**2. Creación de Visualizaciones**

2.1 HEATMAP con px.density_heatmap

Heatmap: day_of_week vs order_status (intensidad = cantidad de pedidos)

In [9]:
df_heat = df.dropna(subset=["order_purchase_timestamp", "order_status"]).copy()

df_heat["day_of_week"] = df_heat["order_purchase_timestamp"].dt.day_name()

# Para que quede ordenado (Lunes → Domingo)
order_days = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
df_heat["day_of_week"] = pd.Categorical(df_heat["day_of_week"], categories=order_days, ordered=True)

heat_data = (
    df_heat.groupby(["day_of_week", "order_status"])
    .size()
    .reset_index(name="orders_count")
)
heat_data.head()

fig_heat = px.density_heatmap(
    heat_data,
    x="day_of_week",
    y="order_status",
    z="orders_count",
    color_continuous_scale="Blues",
    title="Heatmap: Cantidad de pedidos por día de la semana y estado del pedido"
)

fig_heat.update_layout(xaxis_title="Día de la semana", yaxis_title="Estado del pedido")
fig_heat.show()


/tmp/ipython-input-2561012490.py:10: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Para mejorar la interpretación visual del heatmap, se filtran los estados de pedido más relevantes dentro del flujo principal del proceso (approved, shipped, delivered y canceled). Esta decisión permite evitar que estados con baja frecuencia o escasa relevancia analítica distorsionen la escala cromática del gráfico. El resultado facilita la comparación entre etapas clave del proceso y permite identificar patrones temporales en la distribución de pedidos a lo largo de la semana.

In [8]:
valid_status = [
    "approved",
    "shipped",
    "delivered",
    "canceled"
]

heat_data_filtered = heat_data[
    heat_data["order_status"].isin(valid_status)
].copy()


status_order = ["approved", "shipped", "delivered", "canceled"]

heat_data_filtered["order_status"] = pd.Categorical(
    heat_data_filtered["order_status"],
    categories=status_order,
    ordered=True
)


fig_heat = px.density_heatmap(
    heat_data_filtered,
    x="day_of_week",
    y="order_status",
    z="orders_count",
    color_continuous_scale="Blues",
    title="Heatmap: Cantidad de pedidos por día de la semana y estado del pedido"
)

fig_heat.update_layout(
    xaxis_title="Día de la semana",
    yaxis_title="Estado del pedido"
)

fig_heat.show()


El predominio del estado delivered refleja que la mayoría de los pedidos completan exitosamente el proceso, mientras que los estados intermedios o de cancelación presentan una frecuencia significativamente menor.

2.2. Funnel Chart (Gráfico de Embudo)

Vamos a medir cuantos pedidos llegan a cada etapa del proceso:

*   Purchased (compra creada) con **order_purchase_timestamp**
*   Approved (pago aprobado) con **order_approved_at**
*   Handed to Carrier (entregado a logística) con **order_delivered_carrier_date**
*   Delivered to Customer (entregado al cliente) con **order_delivered_customer_date**

Esto forma un funnel real: cada etapa depende de la anterior y normalmente va bajando.

In [12]:
# Crea indicadores por etapa
df_fun = df.copy()

df_fun["stage_purchased"] = df_fun["order_purchase_timestamp"].notna()
df_fun["stage_approved"] = df_fun["order_approved_at"].notna()
df_fun["stage_handed_to_carrier"] = df_fun["order_delivered_carrier_date"].notna()
df_fun["stage_delivered"] = df_fun["order_delivered_customer_date"].notna()

# Cuenta pedidos por etapa
funnel_data = pd.DataFrame({
    "stage": ["Purchased", "Approved", "Handed to Carrier", "Delivered to Customer"],
    "orders_count": [
        df_fun["stage_purchased"].sum(),
        df_fun["stage_approved"].sum(),
        df_fun["stage_handed_to_carrier"].sum(),
        df_fun["stage_delivered"].sum(),
    ]
})

# % respecto a inicio y % respecto a etapa anterior
funnel_data["conversion_pct_vs_start"] = (
    funnel_data["orders_count"] / funnel_data.loc[0, "orders_count"] * 100
).round(2)

funnel_data["conversion_pct_vs_prev"] = (
    funnel_data["orders_count"].div(funnel_data["orders_count"].shift(1)) * 100
).round(2)

# Etiquetas para mostrar dentro del funnel
funnel_data["label"] = (
    funnel_data["orders_count"].astype(str)
    + " ("
    + funnel_data["conversion_pct_vs_start"].astype(str)
    + "%)"
)

# Crea el grafico con Plotly Express
fig_funnel = px.funnel(
    funnel_data,
    x="orders_count",
    y="stage",
    text="label",
    title="Funnel: Progresión de pedidos por etapas del proceso logístico",
    color_discrete_sequence=px.colors.sequential.Blues
)

fig_funnel.update_traces(textposition="inside")
fig_funnel.update_layout(
    xaxis_title="Cantidad de pedidos",
    yaxis_title="Etapa"
)

fig_funnel.show()

# Muestra las tablas
funnel_data



,stage,orders_count,conversion_pct_vs_start,conversion_pct_vs_prev,label
0,Purchased,99441,100.00,NaN,99441 (100.0%)
1,Approved,99281,99.84,99.84,99281 (99.84%)
2,Handed to Carrier,97658,98.21,98.37,97658 (98.21%)
3,Delivered to Customer,96476,97.02,98.79,96476 (97.02%)


El gráfico de embudo muestra la reducción progresiva en la cantidad de pedidos a medida que avanzan por etapas del proceso: compra creada, pago aprobado, entrega al operador logístico y entrega final al cliente.

La disminución entre etapas refleja pérdidas naturales del proceso (por ejemplo, pedidos que no se aprueban, que se cancelan o que no alcanzan la entrega efectiva).

El grafico de embudo muestra una disminucion progresiva en la cantidad de pedidos a medida que avanzan por las distintas etapas del proceso logístico.

Se observa que la mayor parte de los pedidos creados son aprobados exitosamente, lo que indica un bajo nivel de rechazo en la etapa de pago. La principal reduccion se produce entre las etapas de aprobación y entrega al operador logistico, lo que sugiere que las perdidas del proceso se concentran principalmente en fases operativas y logísticas.

Finalmente, la mayoría de los pedidos que alcanzan la etapa de envio logran completarse con una entrega efectiva al cliente.

La alta tasa de conversión entre etapas indica un proceso logístico maduro y estable, donde la mayoría de los pedidos completan exitosamente el ciclo de compra y entrega.

3. Análisis y Conclusiones

**Heatmap **

El heatmap muestra que la mayoría de los pedidos se concentran en el estado delivered a lo largo de todos los días de la semana, lo que indica un proceso logístico estable. No se observan variaciones significativas por día, lo que sugiere una distribución temporal homogénea de los pedidos.

El mapa de calor es adecuado porque permite comparar simultáneamente el estado del pedido y el día de la semana, utilizando la intensidad del color para representar la cantidad de órdenes. Esta codificación visual facilita la identificación rápida de patrones de concentración y estabilidad temporal.

**Funnel Chart**

El funnel evidencia una reduccion progresiva de pedidos entre las distintas etapas del proceso logístico, con una alta tasa de conversión entre la compra y la aprobación del pago, y una mayor pérdida en la etapa de envío. La mayoría de los pedidos que son despachados logran completarse con entrega al cliente.

El funnel chart es efectivo porque representa de forma clara un proceso secuencial de conversion, permitiendo identificar rapidamente en que etapas se producen las principales perdidas y evaluar la eficiencia del flujo logistico.